# Fase 04: Star Schema para Power BI
**Objetivo:** Construir un modelo dimensional optimizado para Power BI a partir de los datos estandarizados en Silver.
**Entrada:** `medicalproyect-processed/silver/`
**Salida:** Star Schema en `medicalproyect-curated/gold/power_bi/`

**Estructura:**
```
dim_paciente ──── fact_ingresos ──── dim_fecha (admission)
     │                              dim_fecha (discharge)
     ├── fact_analiticas  ──── dim_fecha (test_date)
     ├── fact_medicacion  ──── dim_fecha (start_date)
     └── fact_diagnosticos ──── dim_fecha (visit_date)
```

In [ ]:
def _subir_log(sufijo, storage_account):
    try:
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass


# ── Función helper para notificaciones por Telegram ──
def _enviar_telegram(mensaje):
    try:
        from notebookutils import mssparkutils
        import requests
        token = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramToken", NOMBRE_PUENTE)
        chat_id = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramChatId", NOMBRE_PUENTE)
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": mensaje, "parse_mode": "HTML"}, timeout=10)
    except Exception:
        pass

# ── Función helper: sube el log al Data Lake ──
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass

# Configuramos logging para ver el progreso del pipeline
import os
import logging
from datetime import datetime, date, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, year, month, dayofmonth, quarter, dayofweek, when, monotonically_increasing_id

# Limpiamos los handlers de logging previos para evitar duplicados
for handler in logging.root.handlers[:]:
    handler.flush()
    handler.close()
    logging.root.removeHandler(handler)

# Configuramos el logger con formato y salida a archivo y consola
log_filename = "pipeline_star_schema.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_filename, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Medical.StarSchema")

logger.info("=" * 60)
logger.info("CONSTRUYENDO STAR SCHEMA PARA POWER BI")
logger.info("=" * 60)

# Definimos las rutas de los storage accounts de Azure para Silver y Gold
STORAGE_ACCOUNT = "stproyectomastergrupo3"
base_silver = f"abfss://medicalproyect-processed@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver"

# ── Configuración de Key Vault ──
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = "AzureKeyVault"

base_pbi = f"abfss://medicalproyect-curated@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/power_bi"

# Iniciamos la sesion de Spark
spark = SparkSession.builder.appName("Medical_StarSchema").getOrCreate()
logger.info("SparkSession creada.")

# Cargamos todos los DataFrames desde la capa Silver para empezar a construir las dimensiones y hechos
logger.info("Cargando datos desde Silver...")
df_patients = spark.read.parquet(f"{base_silver}/patients")
df_outcomes = spark.read.parquet(f"{base_silver}/outcomes")
df_medications = spark.read.parquet(f"{base_silver}/medications")
df_lab_results = spark.read.parquet(f"{base_silver}/lab_results")
df_diagnoses = spark.read.parquet(f"{base_silver}/diagnoses")
logger.info("Datos cargados correctamente.")

logger.info("📱 Enviando notificación de inicio...")
_enviar_telegram("🚀 INICIADO: 04 Star Schema Powerbi")


## 1. Dimension Paciente
1 fila por paciente. Contiene todos los atributos demograficos y clinicos estaticos.

In [ ]:
logger.info("Construyendo dim_paciente...")

# Creamos la dimension de pacientes con todos sus atributos demograficos y clinicos estaticos
# Esta tabla tiene una fila por paciente y es la tabla central del modelo star schema
dim_paciente = df_patients.select(
    col("patient_id"),
    col("age"),
    col("sex"),
    col("bmi"),
    col("systolic_bp"),
    col("diastolic_bp"),
    col("heart_rate"),
    col("temperature_c"),
    col("smoking_status"),
    col("alcohol_use"),
    col("exercise_level"),
    col("insurance_type"),
    col("charlson_index"),
    col("dx_hypertension"),
    col("dx_type2_diabetes"),
    col("dx_hyperlipidemia"),
    col("dx_obesity"),
    col("dx_coronary_artery_disease"),
    col("dx_heart_failure"),
    col("dx_atrial_fibrillation"),
    col("dx_chronic_kidney_disease"),
    col("dx_copd"),
    col("dx_asthma"),
    col("dx_depression"),
    col("dx_anxiety"),
    col("dx_hypothyroidism"),
    col("dx_osteoarthritis"),
    col("dx_type1_diabetes")
)

# Contamos las filas y guardamos la dimension en formato Parquet en la capa Gold
cnt = dim_paciente.count()
dim_paciente.write.mode("overwrite").parquet(f"{base_pbi}/dim_paciente")
logger.info(f"  dim_paciente: {cnt:,} filas -> {base_pbi}/dim_paciente")

# Subida parcial de log tras esta seccion
_subir_log("powerbi/star_schema_dimension_fecha", STORAGE_ACCOUNT)


## 2. Dimension Fecha
Calendario completo (2018-01-01 a 2025-12-31) con atributos jerarquicos.

In [ ]:
logger.info("Construyendo dim_fecha...")

# Generamos un calendario completo con Spark SQL para poder hacer analisis por tiempo en Power BI
# Creamos un date_id numerico (YYYYMMDD) como clave primaria para relacionar con las tablas de hechos
spark.sql("DROP TABLE IF EXISTS fechas")
spark.sql("""
    CREATE TEMPORARY VIEW fechas AS
    SELECT
        CAST(date_format(fecha, 'yyyyMMdd') AS INT) AS date_id,
        fecha AS full_date,
        YEAR(fecha) AS anio,
        MONTH(fecha) AS mes,
        DAYOFMONTH(fecha) AS dia,
        QUARTER(fecha) AS trimestre,
        DAYOFWEEK(fecha) AS dia_semana,
        date_format(fecha, 'EEEE') AS dia_semana_nombre,
        date_format(fecha, 'MMMM') AS mes_nombre
    FROM (
        SELECT EXPLODE(SEQUENCE(TO_DATE('2018-01-01'), TO_DATE('2025-12-31'), INTERVAL 1 DAY)) AS fecha
    )
""")

# Guardamos la dimension fecha como Parquet
dim_fecha = spark.sql("SELECT * FROM fechas")
cnt_f = dim_fecha.count()
dim_fecha.write.mode("overwrite").parquet(f"{base_pbi}/dim_fecha")
logger.info(f"  dim_fecha: {cnt_f:,} filas -> {base_pbi}/dim_fecha")

# Subida parcial de log tras esta seccion
_subir_log("powerbi/star_schema_ingresos", STORAGE_ACCOUNT)


## 3. Tabla de Hechos: Ingresos Hospitalarios
1 fila por hospitalizacion. Se relaciona con dim_paciente y dim_fecha.

In [ ]:
logger.info("Construyendo fact_ingresos...")

# Creamos la tabla de hechos de hospitalizaciones
# Se relaciona con dim_paciente por patient_id y con dim_fecha por admission_date_id y discharge_date_id
# Contiene metricas como duracion de estancia, costos, readmisiones y si ingresaron a UCI
fact_ingresos = df_outcomes.select(
    col("patient_id"),
    date_format(col("admission_date"), "yyyyMMdd").cast("int").alias("admission_date_id"),
    date_format(col("discharge_date"), "yyyyMMdd").cast("int").alias("discharge_date_id"),
    col("admission_date"),
    col("discharge_date"),
    col("length_of_stay_days"),
    col("icu_admission"),
    col("icu_days"),
    col("in_hospital_death"),
    col("discharge_disposition"),
    col("readmitted_30d"),
    col("days_to_readmission"),
    col("primary_drg"),
    col("total_charges_usd")
)

cnt_i = fact_ingresos.count()
fact_ingresos.write.mode("overwrite").parquet(f"{base_pbi}/fact_ingresos")
logger.info(f"  fact_ingresos: {cnt_i:,} filas -> {base_pbi}/fact_ingresos")

# Subida parcial de log tras esta seccion
_subir_log("powerbi/star_schema_analiticas", STORAGE_ACCOUNT)


## 4. Tabla de Hechos: Analiticas de Laboratorio
~2.8M filas. La tabla mas granular.

In [ ]:
logger.info("Construyendo fact_analiticas...")

# Creamos la tabla de hechos de resultados de laboratorio
# Es la tabla mas granular del modelo (~2.8M filas), con un registro por analitica realizada
# Se relaciona con dim_fecha por test_date_id y con dim_paciente por patient_id
fact_analiticas = df_lab_results.select(
    col("patient_id"),
    date_format(col("test_date"), "yyyyMMdd").cast("int").alias("test_date_id"),
    col("test_date"),
    col("test_name"),
    col("value"),
    col("unit"),
    col("reference_low"),
    col("reference_high"),
    col("flag"),
    col("is_abnormal"),
    col("delta_from_normal")
)

cnt_l = fact_analiticas.count()
fact_analiticas.write.mode("overwrite").parquet(f"{base_pbi}/fact_analiticas")
logger.info(f"  fact_analiticas: {cnt_l:,} filas -> {base_pbi}/fact_analiticas")


## 5. Tabla de Hechos: Medicacion
~364K prescripciones.

In [ ]:
logger.info("Construyendo fact_medicacion...")

# Creamos la tabla de hechos de medicamentos prescritos
# Cada fila es una prescripcion con su dosis, frecuencia, duracion y porcentaje de adherencia
# Se relaciona con dim_paciente por patient_id y con dim_fecha por start_date_id
fact_medicacion = df_medications.select(
    col("patient_id"),
    col("medication"),
    col("dose"),
    col("unit"),
    col("frequency"),
    col("indication"),
    date_format(col("start_date"), "yyyyMMdd").cast("int").alias("start_date_id"),
    col("start_date"),
    col("duration_days"),
    col("is_generic"),
    col("adherence_pct")
)

cnt_m = fact_medicacion.count()
fact_medicacion.write.mode("overwrite").parquet(f"{base_pbi}/fact_medicacion")
logger.info(f"  fact_medicacion: {cnt_m:,} filas -> {base_pbi}/fact_medicacion")


## 6. Tabla de Hechos: Diagnosticos
~274K visitas diagnosticas.

In [ ]:
logger.info("Construyendo fact_diagnosticos...")

# Creamos la tabla de hechos de diagnosticos
# Cada fila representa una visita con su diagnostico principal (ICD-10) y diagnosticos secundarios
# Se relaciona con dim_paciente por patient_id y con dim_fecha por visit_date_id
fact_diagnosticos = df_diagnoses.select(
    col("patient_id"),
    date_format(col("visit_date"), "yyyyMMdd").cast("int").alias("visit_date_id"),
    col("visit_date"),
    col("visit_type"),
    col("primary_diagnosis"),
    col("primary_icd10"),
    col("secondary_diagnoses"),
    col("secondary_icd10s"),
    col("provider_specialty")
)

cnt_d = fact_diagnosticos.count()
fact_diagnosticos.write.mode("overwrite").parquet(f"{base_pbi}/fact_diagnosticos")
logger.info(f"  fact_diagnosticos: {cnt_d:,} filas -> {base_pbi}/fact_diagnosticos")

# Subida parcial de log tras esta seccion
_subir_log("powerbi/star_schema_resumen", STORAGE_ACCOUNT)


## 7. Resumen y Relaciones en Power BI

In [ ]:
# Mostramos el resumen con el conteo de filas de todas las tablas generadas
logger.info("")
logger.info("=" * 60)
logger.info("STAR SCHEMA COMPLETADO")
logger.info("=" * 60)
logger.info(f"  dim_paciente:     {cnt:>7,} filas")
logger.info(f"  dim_fecha:        {cnt_f:>7,} filas")
logger.info(f"  fact_ingresos:    {cnt_i:>7,} filas")
logger.info(f"  fact_analiticas:  {cnt_l:>7,} filas")
logger.info(f"  fact_medicacion:  {cnt_m:>7,} filas")
logger.info(f"  fact_diagnosticos:{cnt_d:>7,} filas")
# Imprimimos las instrucciones de las relaciones que debemos crear manualmente en Power BI
logger.info("")
logger.info("Relaciones a crear en Power BI:")
logger.info("  dim_paciente[patient_id] 1:N fact_ingresos[patient_id]")
logger.info("  dim_paciente[patient_id] 1:N fact_analiticas[patient_id]")
logger.info("  dim_paciente[patient_id] 1:N fact_medicacion[patient_id]")
logger.info("  dim_paciente[patient_id] 1:N fact_diagnosticos[patient_id]")
logger.info("  dim_fecha[date_id] 1:N fact_ingresos[admission_date_id]")
logger.info("  dim_fecha[date_id] 1:N fact_ingresos[discharge_date_id]")
logger.info("  dim_fecha[date_id] 1:N fact_analiticas[test_date_id]")
logger.info("  dim_fecha[date_id] 1:N fact_medicacion[start_date_id]")
logger.info("  dim_fecha[date_id] 1:N fact_diagnosticos[visit_date_id]")
# Copiamos el log del pipeline al storage de Azure para tener trazabilidad
logger.info("")
logger.info("Guardando logs...")

try:
    from notebookutils import mssparkutils
    CONTAINER_LOGS = "medicalproyect-logs"
    log_destino = f"abfss://{CONTAINER_LOGS}@{STORAGE_ACCOUNT}.dfs.core.windows.net/powerbi/star_schema_{datetime.now().strftime('%Y%m%d_%H%M')}.log"
    ruta_local = f"file://{os.path.abspath(log_filename)}"
    mssparkutils.fs.cp(ruta_local, log_destino)
    logger.info(f"Log persistido en: {log_destino}")
except Exception as e:
    logger.warning(f"No se pudo copiar el log: {e}")

# Cerramos la sesion de Spark para liberar recursos
logger.info("EXECUTION STATUS: SUCCESS")
_enviar_telegram("✅ FINALIZADO: 04 Star Schema Powerbi")
spark.stop()
